# Single-Day Hydrogen Model Debug

This notebook is for model debugging only. Do not use it as final thesis evidence.

It runs one day and compares:
- `price_insensitive`
- `stochastic_risk_neutral`
- `stochastic_cvar`
- `perfect_foresight`

## Configuration Snippet

Edit only these values first:
- `custom_start` / `custom_end`: single target day
- model in `models.include`
- `risk.gamma`
- `hydrogen_system.reserve_fraction`

Physical checks in this notebook verify reserve, ramp, and energy balance equations.

In [ ]:
from pathlib import Path
import sys
from dataclasses import replace

repo_root = Path.cwd()
while repo_root.name != "Thesis" and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "scripts" / "Data" / "03_Hydrogen_Test_Case"))

from hydrogen.plant_parameters import load_hydrogen_config
from hydrogen.rolling_horizon import run_hydrogen_backtest

config = load_hydrogen_config(repo_root / "scripts" / "Data" / "03_Hydrogen_Test_Case" / "configs" / "base_hydrogen.yaml")
config = replace(
    config,
    experiment=replace(config.experiment,
                       execution_mode="custom_period",
                       custom_start="2024-10-01",
                       custom_end="2024-10-07"),
    risk=replace(config.risk, gamma=0.05),
    hydrogen_system=replace(config.hydrogen_system, reserve_fraction=0.10),
)
result = run_hydrogen_backtest(config)
result["summary"]

## Figure Interpretation

- Price and operation figure: checks whether consumption shifts toward lower-price periods.
- Buffer trajectory: reserve and inventory feasibility check.
- Strategy dashboard: relative economics and risk.

If reserve violations or mass-balance errors appear in `validation_checks.csv`, stop and fix before larger runs.

In [ ]:
result["validation_checks"].head(30)